# EPV and ECM calculation:
**Task:** 
1. Construct the molecule
2. Get the wave function (both at NR an REL levels)
3. Calculate ECM and Epv
4. Manipulate data
5. Get Molecular orbitals composition (at NR level)

In [ ]:
from pyECM.molecule_class import molecula
import numpy as np
from pyscf import lib
from pyscf.lib.misc import light_speed

cvalue = 137.03599967994*10

with light_speed(cvalue):
    c = lib.param.LIGHT_SPEED


    MOL = 'CFMAR'
    data_folder='/home/juanjoaucar/Documentos/ECM/pyECM2/pyECM/data/'
    basis='sto6g'

    mymolecule = molecula(XYZ_file = data_folder + 'import/' + MOL + '_chiral.xyz', XYZ_achiral_file = data_folder + 'import/' + MOL + '_achiral_S20.xyz')

    # Optional: generate xyz files
    #mymolecule.export_xyz(folder=data_folder+ 'export/', prefix_name=MOL+'_chiral', DIRAC = True, achiral= data_folder + 'import/' + MOL + '_achiral_S20.xyz')

    # Calculate CCMs
    mymolecule.CCM()

    #Define options for the Gaussian Type Orbitals (generated by pySCF) and the WF calculation.
    GTO={'basis': basis, 'charge' : 0, 'spin' : 0, 'verbose' : 0}
    WF_method = {'fourcomp': True, 'debug' : 0, 'cvalue' : c}

    # Get the WF with the pySCF code
    mymolecule.pySCF_WF(gto_dict=GTO, method_dict=WF_method)

    #Calculate ECM
    method = {'fourcomp': True, 'debug' : 0, 'cvalue' : cvalue}
    mymolecule.ECM(method_dict=method)

    #Get Epv
    mymolecule.Epv()

    #Get Epv (using the density matrix)
    density_matrix = mymolecule.rel_pyscf.make_rdm1()
    mymolecule.Epv(dm = density_matrix)

    #Print section
    print(mymolecule.CCM1, mymolecule.CCM2, mymolecule.ECM_NR, mymolecule.ECM_4c)


In [ ]:
#Shape of Epv array
print(mymolecule.Epv_expval.shape)

print("Epv contribution from nucleus of index 0:")
print(np.sum(mymolecule.Epv_expval, axis=1)[0])

print("Orbital contributions to ECM-NR, ECM-4c, Epv-4c:")
for i in range (mymolecule.rel_Noccupied_MO):
    ECM_4c_orbital = mymolecule.ECM_4c_molcontr[i]
    Epv_4c_orbital = np.sum(mymolecule.Epv_expval, axis=0)[i]
    print(mymolecule.ECM_NR_molcontr[i],ECM_4c_orbital,Epv_4c_orbital)


In [ ]:
# Export data to HDF5 file.
import h5py
with h5py.File(MOL+'.h5', 'w') as file:
    file.create_dataset('ECM_4c', data=mymolecule.ECM_4c_molcontr)
    file.create_dataset('ECM_NR', data=mymolecule.ECM_NR_molcontr)
    file.create_dataset('Epv_4c', data=np.sum(mymolecule.Epv_expval, axis=0))

In [ ]:
import h5py
# Read data from HDF5 file.
with h5py.File(MOL+'.h5', 'r') as file:
    ECM_4c_orbital_hdf5 = file['ECM_4c'][:]
    ECM_NR_orbital_hdf5 = file['ECM_NR'][:]
    Epv_4c_orbital_hdf5 = file['Epv_4c'][:]

print("Shape ECM (4c):", ECM_4c_orbital_hdf5.shape)
print("Shape ECM (NR):", ECM_NR_orbital_hdf5.shape)
print("Shape Epv (4c):", Epv_4c_orbital_hdf5.shape)

In [ ]:
from pyscf.tools import mo_mapping

print("AOs:",mymolecule.AO_number)
print("Electrons:",mymolecule.NR_pyscf.mol.nelec)
print("Each Molecular Orbital is doubly occupied")

comp = mo_mapping.mo_comps('Cl 1s', mymolecule.NR_pyscf.mol, mymolecule.NR_pyscf.mo_coeff)
print('MO-id   Cl-1s components')
for i,c in enumerate(comp):
    print('%-3d      %.4f' % (i, c))